In [18]:
#%pip install requests pandas tqdm unidecode numpy

In [19]:
import requests
import pandas as pd
import os
import time
import uuid
import re
from pathlib import Path
from tqdm import tqdm
from unidecode import unidecode
import numpy as np

# Define paths
current_path = Path.cwd()

# Find the project root by looking for 'fpl_pipeline' in the path
# or moving up until we find the 'data' folder
project_root = current_path
while project_root.name != 'fpl_pipeline' and project_root.parent != project_root:
    project_root = project_root.parent

# Set paths relative to the project root
DATA_ROOT = project_root / "data" / "2025-26"
GW_DIR = DATA_ROOT / "gws"
OUTPUT_DIR = project_root / "output"

print(f"Checking existing folders:")
print(f"Project Root: {project_root.resolve()}")
print(f"Data Path: {DATA_ROOT.resolve()}")
print(f"Output Path: {OUTPUT_DIR.resolve()}")

# Double check if these folders actually exist before proceeding
if not DATA_ROOT.exists():
    print(f"Warning: Data folder not found at {DATA_ROOT}. Check your folder structure.")
if not OUTPUT_DIR.exists():
    print(f"Warning: Output folder not found at {OUTPUT_DIR}. Check your folder structure.")

# Ensure directories exist (it won't overwrite existing files)
os.makedirs(GW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://fantasy.premierleague.com/api/"

Checking existing folders:
Project Root: C:\Users\skourako\diplo\fpl_pipeline
Data Path: C:\Users\skourako\diplo\fpl_pipeline\data\2025-26
Output Path: C:\Users\skourako\diplo\fpl_pipeline\output


In [20]:
# Cell 2: Fetch Raw Static Data (Teams, Players, and Fixtures)
def fetch_static_data():
    print("Fetching bootstrap-static and fixtures...")
    bootstrap = requests.get(BASE_URL + "bootstrap-static/").json()
    
    # 1. Teams
    teams = [{"id": t["id"], "name": t["name"], "short_name": t["short_name"]} for t in bootstrap["teams"]]
    pd.DataFrame(teams).to_csv(DATA_ROOT / "teams.csv", index=False)
    
    # 2. Players Raw
    players = pd.DataFrame(bootstrap["elements"])
    keep_p = ["id", "team", "element_type", "first_name", "second_name", "web_name"]
    players[keep_p].rename(columns={"id": "element"}).to_csv(DATA_ROOT / "players_raw.csv", index=False)
    
    # 3. Fixtures
    fx_data = requests.get(BASE_URL + "fixtures/").json()
    fx_df = pd.DataFrame(fx_data)
    keep_f = ["id", "event", "team_h", "team_a", "team_h_difficulty", "team_a_difficulty", "kickoff_time"]
    fx_df[keep_f].to_csv(DATA_ROOT / "fixtures.csv", index=False)
    
    print(" Static files updated in data/2025-26/")
    return bootstrap["events"]

events = fetch_static_data()

Fetching bootstrap-static and fixtures...
 Static files updated in data/2025-26/


In [21]:
#Cell 3: Fetch Completed Gameweek Live Data
def fetch_live_gws(events_list):
    available_gws = [e["id"] for e in events_list if e["finished"] or e["data_checked"]]
    players_raw = pd.read_csv(DATA_ROOT / "players_raw.csv")
    fixtures = pd.read_csv(DATA_ROOT / "fixtures.csv")

    for gw in tqdm(available_gws, desc="Downloading GWs"):
        r = requests.get(f"{BASE_URL}event/{gw}/live/").json()
        rows = []
        for p in r["elements"]:
            stats = p["stats"]
            stats["element"] = p["id"]
            rows.append(stats)
        
        df = pd.DataFrame(rows)
        df["Gameweek"] = gw
        df["season"] = "2025-26"
        
        # Add basic info
        players_raw["name"] = players_raw["first_name"] + " " + players_raw["second_name"]
        df = df.merge(players_raw[["element", "name", "team"]], on="element", how="left")
        
        # Opponent reconstruction
        gw_fx = fixtures[fixtures["event"] == gw]
        opp_rows = []
        for _, f in gw_fx.iterrows():
            opp_rows.append({"team": f["team_h"], "opponent_team": f["team_a"], "was_home": True})
            opp_rows.append({"team": f["team_a"], "opponent_team": f["team_h"], "was_home": False})
        
        df = df.merge(pd.DataFrame(opp_rows), on="team", how="left")
        # Save to the gws subfolder
        out_path = GW_DIR / f"gw{gw}.csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")

fetch_live_gws(events)

In [22]:
#Cell 4: Processing and Feature Engineering (The Logic Engine)
def normalize_player_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip().lower()
    name = unidecode(name)
    name = re.sub(r"[\s_]*\d+\s*$", "", name)
    name = name.replace("_", " ")
    name = "".join(c for c in name if c.isalnum() or c.isspace())
    name = " ".join(name.split())
    return name

# 1. Load and combine all GW files
print("Combining gameweek files...")
gw_files = sorted(
   [f for f in os.listdir(GW_DIR) if f.startswith("gw") and f.endswith(".csv")],
    key=lambda x: int(x.replace("gw", "").replace(".csv", ""))
)

if not gw_files:
    print("No GW files found to process.")
else:
    df_gw = pd.concat([pd.read_csv(GW_DIR / f) for f in gw_files], ignore_index=True)

    # 2. Build Fixture/Difficulty Map
    fixtures_df = pd.read_csv(DATA_ROOT / "fixtures.csv")
    rows = []
    for _, r in fixtures_df.iterrows():
        if pd.isna(r["event"]): continue
        gw = int(r["event"])
        h, a = int(r["team_h"]), int(r["team_a"])
        dh, da = int(r["team_h_difficulty"]), int(r["team_a_difficulty"])
        rows.append({"Gameweek": gw, "opponent_team": a, "Player Team ID": h, "Is Home": True, "Opponent Difficulty": dh})
        rows.append({"Gameweek": gw, "opponent_team": h, "Player Team ID": a, "Is Home": False, "Opponent Difficulty": da})
    fixture_map = pd.DataFrame(rows)

    # 3. Merge and Clean
    df = df_gw.merge(fixture_map, on=["Gameweek", "opponent_team"], how="left")
    
    players_raw = pd.read_csv(DATA_ROOT / "players_raw.csv")
    POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
    df = df.merge(players_raw[["element", "element_type"]], on="element", how="left")
    df["Position"] = df["element_type"].map(POS_MAP)
    df["Player Name Norm"] = df["name"].apply(normalize_player_name)
    
    rename_map = {
        "minutes": "Minutes Played", "total_points": "Total Points",
        "goals_scored": "Goals Scored", "assists": "Assists",
        "clean_sheets": "Clean Sheet", "goals_conceded": "Goals Conceded",
        "ict_index": "ICT Index"
    }
    df.rename(columns=rename_map, inplace=True)

    # 4. Feature Engineering: Injuries, Lags, and UUIDs
    df = df.sort_values(["Player Name Norm", "season", "Gameweek"])
    
    # Injury flag (3 games with 0 mins)
    df["Injury/Unavailable"] = df.groupby("Player Name Norm")["Minutes Played"].transform(
        lambda x: x.rolling(window=3, min_periods=3).apply(lambda s: 1 if s.sum() == 0 else 0, raw=True)
    ).fillna(0)

    # Rolling averages (3 and 5 weeks)
    metrics = ["Total Points", "Minutes Played", "Goals Scored", "Assists"]
    for w in [3, 5]:
        for c in metrics:
            df[f"Avg_{c}_L{w}"] = df.groupby("Player Name Norm")[c].transform(
                lambda s: s.shift(1).rolling(w, min_periods=1).mean()
            ).fillna(0)

    # UUID Maintenance
    mapping_path = OUTPUT_DIR / "player_uuid_mapping.csv"
    if mapping_path.exists():
        map_df = pd.read_csv(mapping_path)
        uuid_map = dict(zip(map_df["Player Name Norm"].astype(str), map_df["Player UUID"].astype(str)))
    else:
        uuid_map = {}
    
    def get_uuid(name):
        if name not in uuid_map: uuid_map[name] = str(uuid.uuid4())
        return uuid_map[name]
    
    df["Player UUID"] = df["Player Name Norm"].apply(get_uuid)
    pd.DataFrame(list(uuid_map.items()), columns=["Player Name Norm", "Player UUID"]).to_csv(mapping_path, index=False)

    # Team Contribution
    df["Team_Points_Contribution_GW_Pct"] = (df["Total Points"] / df.groupby(["team", "Gameweek"])["Total Points"].transform("sum") * 100).fillna(0).round(2)

    # 5. Save final file
    final_path = OUTPUT_DIR / "training_data_2025_26.csv"
    df.to_csv(final_path, index=False, encoding="utf-8-sig")
    print(f"Final training data saved to: {final_path}")
    print(df.head())

Combining gameweek files...
Final training data saved to: c:\Users\skourako\diplo\fpl_pipeline\output\training_data_2025_26.csv
      Minutes Played  Goals Scored  Assists  Clean Sheet  Goals Conceded  \
233                0             0        0            0               0   
923                0             0        0            0               0   
1628               0             0        0            0               0   
2340               0             0        0            0               0   
3080               0             0        0            0               0   

      own_goals  penalties_saved  penalties_missed  yellow_cards  red_cards  \
233           0                0                 0             0          0   
923           0                0                 0             0          0   
1628          0                0                 0             0          0   
2340          0                0                 0             0          0   
3080          0     